# 06 — Normal reference expansion audit

本 Notebook 只做第二阶段第 1 步：系统盘点 NPPAD 中可以合法作为 Normal reference 的来源。

审计原则：
- 只把明确标注为 Normal 的独立轨迹列为 A/B；
- 事故轨迹只能按 Transient Report 的首个 `Malfunction` 时间截取事故前元数据，绝不把事故后数据标为 Normal；
- 对事故前窗口要求至少有两个严格早于注入时刻的采样点且窗口有正时长；
- 训练、校准、测试按完整轨迹分离，不按同一轨迹的行随机切分。

分级：A = 可直接纳入独立 Normal reference；B = 可用于不同工况/变功率辅助建模但需谨慎；C = 不可合法作为 Normal reference。

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path(r'C:/Users/18205/NPP-Guard')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from normal_reference import write_audit_results

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 180)

## 1. 全量扫描并写出审计结果

扫描包括原始 `NPPAD/`、`Operation_csv_data/`、`Variable_Power_Data/`、两个 README、Transient Report 和变量功率 Normal MDB。事故 MDB 不逐个重复读取：事故 CSV 是同一 `PlotData` 的导出，本次只需要用它完成事故前时间轴/97 列/初始状态审计；固定 Normal 原始 MDB 和 4 个 NORM MDB 会实际通过 Access ODBC 读取。

In [2]:
inventory, summary, audit = write_audit_results(
    result_root=PROJECT_ROOT / 'results',
    data_root=PROJECT_ROOT / 'data' / 'NuclearPowerPlantAccidentData',
)

print('inventory rows:', len(inventory))
print('summary rows:', len(summary))
print('output files:')
for name in (
    'normal_reference_inventory.csv',
    'normal_reference_audit_summary.csv',
    'normal_reference_audit_summary.json',
):
    path = PROJECT_ROOT / 'results' / name
    print(f'  {path} ({path.stat().st_size:,} bytes)')

inventory rows: 1221
summary rows: 21
output files:
  C:\Users\18205\NPP-Guard\results\normal_reference_inventory.csv (1,138,572 bytes)
  C:\Users\18205\NPP-Guard\results\normal_reference_audit_summary.csv (4,114 bytes)
  C:\Users\18205\NPP-Guard\results\normal_reference_audit_summary.json (17,830 bytes)


## 2. Normal candidate 分级

In [3]:
normal_candidates = inventory[inventory['is_legal_normal_reference']].copy()
normal_candidates[[
    'classification', 'source_kind', 'scenario_id', 'source_path',
    'rows', 'time_start_s', 'time_end_s', 'sampling_median_s',
    'nonuniform_sampling_intervals', 'pwr_start_pct', 'pwr_end_pct',
    'pwr_min_pct', 'pwr_max_pct', 'reference_97_variables_available',
    'loca_initial_condition_match', 'mdb_read_status',
    'train_use', 'calibration_use', 'test_use'
]]

,classification,source_kind,scenario_id,source_path,rows,time_start_s,time_end_s,sampling_median_s,nonuniform_sampling_intervals,pwr_start_pct,pwr_end_pct,pwr_min_pct,pwr_max_pct,reference_97_variables_available,loca_initial_condition_match,mdb_read_status,train_use,calibration_use,test_use
1216,A,fixed_normal_csv,NPPAD_NORMAL_1,data/NuclearPowerPlantAccidentData/Operation_c...,302,0.0,3010.0,10.0,0,100.0,40.090614,38.548420,100.000000,True,True,success,"yes, as the sole direct Normal training source",no independent calibration trajectory available,no independent Normal test trajectory available
1217,B,variable_power_normal_mdb,NORM_100_to_80,data/NuclearPowerPlantAccidentData/Variable_Po...,375,0.0,3740.0,10.0,0,100.0,80.061996,79.596481,100.336029,True,True,success,"yes, condition-aware model only",yes only at trajectory level; never random row...,yes only as a held-out condition-aware trajectory
1218,B,variable_power_normal_mdb,NORM_70_to_90,data/NuclearPowerPlantAccidentData/Variable_Po...,258,0.0,3286.0,10.0,1,100.0,90.230400,69.906731,100.000000,True,True,success,"yes, condition-aware model only",yes only at trajectory level; never random row...,yes only as a held-out condition-aware trajectory
1219,B,variable_power_normal_mdb,NORM_80_to_100,data/NuclearPowerPlantAccidentData/Variable_Po...,228,0.0,3630.0,10.0,1,100.0,100.419296,79.882477,100.634392,True,True,success,"yes, condition-aware model only",yes only at trajectory level; never random row...,yes only as a held-out condition-aware trajectory
1220,B,variable_power_normal_mdb,NORM_90_to_70,data/NuclearPowerPlantAccidentData/Variable_Po...,287,0.0,3375.5,10.0,1,100.0,69.993462,69.645081,100.000000,True,True,success,"yes, condition-aware model only",yes only at trajectory level; never random row...,yes only as a held-out condition-aware trajectory


结论性解释：固定 `Normal/1.csv` 是唯一 A 级直接来源，但它的 `PWR/PWNT` 从名义 100% 下降到约 40%，因此不能把它称为全程固定功率稳定参考。4 个 `NORM_*` MDB 明确是正常功率过渡轨迹，归为 B 级；其中若存在大时间跳跃，后续建模必须保留轨迹边界并按完整轨迹分割。

## 3. 所有事故类别的注入时间与事故前窗口

In [4]:
accident = inventory[inventory['source_group'].eq('accident_pre_injection')].copy()
accident_summary = summary[summary['summary_group'].eq('accident_pre_injection')].copy()
accident_summary[[
    'label', 'trajectory_count', 'injection_min_s', 'injection_max_s',
    'pre_rows_min', 'pre_rows_max', 'pre_window_min_s', 'pre_window_max_s',
    'usable_pre_window_count', 'reference_97_variables_all',
    'schema_97_all', 'initial_match_count'
]]

,label,trajectory_count,injection_min_s,injection_max_s,pre_rows_min,pre_rows_max,pre_window_min_s,pre_window_max_s,usable_pre_window_count,reference_97_variables_all,schema_97_all,initial_match_count
2,accident pre-injection audit,1216,0.5,0.5,1.0,1.0,0.0,0.0,0,True,False,1191
3,ATWS,1,0.5,0.5,1.0,1.0,0.0,0.0,0,True,True,1
4,FLB,100,0.5,0.5,1.0,1.0,0.0,0.0,0,True,True,100
5,LACP,1,0.5,0.5,1.0,1.0,0.0,0.0,0,True,True,1
6,LLB,101,0.5,0.5,1.0,1.0,0.0,0.0,0,True,True,101
7,LOCA,100,0.5,0.5,1.0,1.0,0.0,0.0,0,True,True,100
8,LOCAC,100,0.5,0.5,1.0,1.0,0.0,0.0,0,True,True,100
9,LOF,1,0.5,0.5,1.0,1.0,0.0,0.0,0,True,True,1
10,LR,99,0.5,0.5,1.0,1.0,0.0,0.0,0,True,True,99
11,MD,100,0.5,0.5,1.0,1.0,0.0,0.0,0,True,True,100


In [5]:
print('unique accident injection times:', sorted(accident['injection_time_s'].dropna().unique()))
print('pre-injection row counts:', sorted(accident['pre_injection_rows'].dropna().unique()))
print('pre-injection window lengths:', sorted(accident['pre_injection_window_s'].dropna().unique()))
print('usable accident pre-windows:', int(accident['pre_window_usable_for_normal'].sum()))

schema_exceptions = inventory[
    (inventory['schema_extra_columns'].fillna('') != '')
    | (inventory['schema_missing_columns'].fillna('') != '')
][['source_category', 'case_id', 'column_count', 'schema_extra_columns', 'schema_missing_columns', 'loca_initial_condition_match']]
schema_exceptions.head(30)

unique accident injection times: [np.float64(0.5)]
pre-injection row counts: [np.float64(1.0)]
pre-injection window lengths: [np.float64(0.0)]
usable accident pre-windows: 0


,source_category,case_id,column_count,schema_extra_columns,schema_missing_columns,loca_initial_condition_match
1013,SLBIC,1,100,WPCS;WPFW;WPMU,,False
1014,SLBIC,10,100,WPCS;WPFW;WPMU,,False
1017,SLBIC,11,100,WPCS;WPFW;WPMU,,False
1018,SLBIC,12,100,WPCS;WPFW;WPMU,,False
1019,SLBIC,13,100,WPCS;WPFW;WPMU,,False
1020,SLBIC,14,100,WPCS;WPFW;WPMU,,False
1021,SLBIC,15,100,WPCS;WPFW;WPMU,,False
1022,SLBIC,16,100,WPCS;WPFW;WPMU,,False
1023,SLBIC,17,100,WPCS;WPFW;WPMU,,False
1024,SLBIC,18,100,WPCS;WPFW;WPMU,,False


事故审计结果的关键点是：17 个事故类别、1216 个事故案例的首个明确事故注入全部为 `0.5 s`；CSV 采样间隔为 `10 s`，因此每个案例只有 `t=0` 一个严格事故前采样点，事故前窗口长度为 `0 s`。这些行保留在 inventory 中作为证据，但全部是 C 级，不能扩充 Normal reference。

## 4. 最终决策

In [6]:
print(audit['conclusion']['recommendation'])
print('A direct:', audit['normal_reference_counts']['A_direct_normal'])
print('B auxiliary:', audit['normal_reference_counts']['B_auxiliary_normal'])
print('C not legal:', audit['normal_reference_counts']['C_not_legal_normal'])
print('variable-power MDB successes:', audit['mdb_validation']['variable_power_normal_mdb_success_count'])

assert audit['conclusion']['rigorous_loca_early_warning_supported'] is False
assert audit['normal_reference_counts']['A_direct_normal'] == 1
assert audit['normal_reference_counts']['B_auxiliary_normal'] == 4
assert audit['normal_reference_counts']['C_not_legal_normal'] == len(accident)
assert set(accident['injection_time_s'].dropna()) == {0.5}
assert set(accident['pre_injection_rows'].dropna()) == {1}
assert set(accident['pre_injection_window_s'].dropna()) == {0.0}
assert int(accident['pre_window_usable_for_normal'].sum()) == 0
assert audit['mdb_validation']['fixed_normal_mdb_matches_csv'] is True
assert audit['mdb_validation']['variable_power_normal_mdb_success_count'] == 4
print('FULL NORMAL-REFERENCE AUDIT PASSED')

Do not make strict LOCA early warning the main Phase-2 line. Keep the five normal trajectories for condition-aware exploratory work, but shift the main line to LOCA severity estimation and/or protection-time prediction until independent full-power Normal runs are available.
A direct: 1
B auxiliary: 4
C not legal: 1216
variable-power MDB successes: 4
FULL NORMAL-REFERENCE AUDIT PASSED
